# Skin Condition Detection — Production Training Pipeline (PyTorch)

**Improvements over v1:**
- ✅ Google Colab support with auto-extract from `/content/dataset.zip`
- ✅ EfficientNet-B2 backbone (stronger than B0, still fast)
- ✅ Enhanced augmentation: RandAugment, GaussianBlur, RandomAffine, MixUp
- ✅ Focal Loss for minority-class robustness
- ✅ Cosine annealing with linear warm-up scheduler
- ✅ Fixed deprecated `torch.cuda.amp` APIs
- ✅ Confusion matrix + full classification report
- ✅ One-click Colab model download
- ✅ Test-Time Augmentation (TTA) in inference helper


In [1]:
# ─────────────────────────────────────────────────────────────────────────
# COLAB FLAG  →  set IS_COLAB = True when running on Google Colab
# ─────────────────────────────────────────────────────────────────────────
IS_COLAB = False   # <── flip to True on Colab

if IS_COLAB:
    import zipfile
    from pathlib import Path as _P

    _ZIP  = _P('/content/dataset.zip')
    _DEST = _P('/content/dataset')

    if not _ZIP.exists():
        raise FileNotFoundError(
            'dataset.zip not found at /content/dataset.zip.\n'
            'Upload it via the Colab Files panel or Google Drive first.'
        )

    if _DEST.exists():
        print(f'Already extracted → {_DEST}, skipping.')
    else:
        print(f'Extracting {_ZIP} ...')
        with zipfile.ZipFile(_ZIP, 'r') as zf:
            zf.extractall(_DEST)
        print('Done.')

    # Handle zip with a single top-level folder inside
    _subdirs = [p for p in _DEST.iterdir() if p.is_dir()]
    if len(_subdirs) == 1 and not (_DEST / 'Acne').exists():
        _DATASET_ROOT = str(_subdirs[0])
        print(f'Using inner folder: {_DATASET_ROOT}')
    else:
        _DATASET_ROOT = str(_DEST)

    OUTPUT_DIR = '/content/artifacts'
else:
    _DATASET_ROOT = '/home/dev/Desktop/cnn/dataset'   # <── local path
    OUTPUT_DIR    = '/home/dev/Desktop/cnn/artifacts'

print('Dataset root :', _DATASET_ROOT)
print('Output dir   :', OUTPUT_DIR)


Dataset root : /home/dev/Desktop/cnn/dataset
Output dir   : /home/dev/Desktop/cnn/artifacts


In [2]:
# !pip install -q torch torchvision pandas numpy matplotlib tqdm pillow scikit-learn

import os, json, time, copy, math, random
from dataclasses import dataclass, asdict, field
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms, models

try:
    from sklearn.metrics import classification_report, confusion_matrix
    import seaborn as sns
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False
    print('scikit-learn / seaborn not found — install for full report.')

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


PyTorch: 2.10.0+cu128
CUDA available: False


In [3]:
@dataclass
class Config:
    # Paths
    dataset_root: str = _DATASET_ROOT
    output_dir:   str = OUTPUT_DIR
    run_name:     str = 'skin_efficientnet_b2'

    # Image / loader
    img_size:    int = 260   # native EfficientNet-B2 resolution
    batch_size:  int = 32
    num_workers: int = 2

    # Training
    epochs:        int   = 40
    lr:            float = 3e-4
    weight_decay:  float = 1e-4
    label_smoothing: float = 0.05
    warmup_epochs: int   = 3

    # Splits
    train_ratio: float = 0.75
    val_ratio:   float = 0.15
    test_ratio:  float = 0.10

    # Regularisation
    mixup_alpha:        float = 0.3    # 0 → disabled
    freeze_backbone_epochs: int = 2

    # Imbalance handling
    use_weighted_sampler:    bool = True
    use_focal_loss:          bool = True   # replaces standard CE when True
    focal_gamma:             float = 2.0
    use_class_weighted_loss: bool = True

    # Early stopping
    patience:  int   = 8
    min_delta: float = 1e-4

    seed: int = 42


cfg = Config()
assert abs((cfg.train_ratio + cfg.val_ratio + cfg.test_ratio) - 1.0) < 1e-8

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(cfg.seed)

RUN_DIR = Path(cfg.output_dir) / cfg.run_name
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Run directory:', RUN_DIR)


Device: cpu
Run directory: /home/dev/Desktop/cnn/artifacts/skin_efficientnet_b2


In [4]:
# ── Transforms ────────────────────────────────────────────────────────────
# Strong augmentation for training (skin images benefit from colour + spatial aug)

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(cfg.img_size, scale=(0.6, 1.0), ratio=(0.8, 1.25)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.15),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=10),
    # Skin-aware colour jitter (more saturation/hue for lesion colour variance)
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    # RandAugment gives diverse policy-search style augmentation
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15), ratio=(0.3, 3.3), value='random'),
])

eval_tfms = transforms.Compose([
    transforms.Resize(int(cfg.img_size * 1.14)),
    transforms.CenterCrop(cfg.img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

# TTA (Test-Time Augmentation) transform set used at inference
tta_tfms = [
    eval_tfms,  # original
    transforms.Compose([
        transforms.Resize(int(cfg.img_size * 1.14)),
        transforms.CenterCrop(cfg.img_size),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]),
    transforms.Compose([
        transforms.Resize(int(cfg.img_size * 1.25)),
        transforms.CenterCrop(cfg.img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]),
]

print('Augmentation pipeline ready.')
print(f'  Training  transforms: {len(train_tfms.transforms)} steps')
print(f'  Eval/Test transforms: {len(eval_tfms.transforms)} steps')
print(f'  TTA views           : {len(tta_tfms)}')


Augmentation pipeline ready.
  Training  transforms: 12 steps
  Eval/Test transforms: 4 steps
  TTA views           : 3


In [5]:
base_dataset = datasets.ImageFolder(cfg.dataset_root)
class_names  = base_dataset.classes
num_classes  = len(class_names)
print('Classes:', class_names)
print('Num classes:', num_classes)

with open(RUN_DIR / 'class_to_idx.json', 'w') as f:
    json.dump(base_dataset.class_to_idx, f, indent=2)

# ── Stratified split ────────────────────────────────────────────────────────
targets = np.array(base_dataset.targets)
indices = np.arange(len(base_dataset))

by_class = defaultdict(list)
for idx, y in zip(indices, targets):
    by_class[int(y)].append(int(idx))

train_idx, val_idx, test_idx = [], [], []
for cls, cls_indices in by_class.items():
    cls_indices = np.array(cls_indices)
    rng = np.random.default_rng(cfg.seed + cls)
    rng.shuffle(cls_indices)
    n = len(cls_indices)
    if n == 1:
        n_train, n_val, n_test = 1, 0, 0
    elif n == 2:
        n_train, n_val, n_test = 1, 0, 1
    else:
        n_train = max(1, int(round(n * cfg.train_ratio)))
        n_val   = max(1, int(round(n * cfg.val_ratio)))
        n_test  = n - n_train - n_val
        if n_test < 1:
            deficit = 1 - n_test
            take = min(deficit, max(0, n_train - 1))
            n_train -= take; deficit -= take
            take = min(deficit, max(0, n_val - 1))
            n_val -= take
            n_test = n - n_train - n_val
    train_idx.extend(cls_indices[:n_train].tolist())
    val_idx.extend(cls_indices[n_train:n_train + n_val].tolist())
    test_idx.extend(cls_indices[n_train + n_val:n_train + n_val + n_test].tolist())

if not val_idx:  val_idx.append(train_idx.pop())
if not test_idx: test_idx.append(train_idx.pop())

print(f'Train/Val/Test sizes: {len(train_idx)}, {len(val_idx)}, {len(test_idx)}')

train_dataset_full = datasets.ImageFolder(cfg.dataset_root, transform=train_tfms)
val_dataset_full   = datasets.ImageFolder(cfg.dataset_root, transform=eval_tfms)
test_dataset_full  = datasets.ImageFolder(cfg.dataset_root, transform=eval_tfms)

train_subset = Subset(train_dataset_full, train_idx)
val_subset   = Subset(val_dataset_full,   val_idx)
test_subset  = Subset(test_dataset_full,  test_idx)

train_targets = targets[train_idx]
val_targets   = targets[val_idx]
test_targets  = targets[test_idx]

dist_df = pd.DataFrame({
    'class': class_names,
    'train': np.bincount(train_targets, minlength=num_classes),
    'val':   np.bincount(val_targets,   minlength=num_classes),
    'test':  np.bincount(test_targets,  minlength=num_classes),
})
display(dist_df)


Classes: ['Acne', 'Acne_Blackheads', 'Acne_Cystic', 'Acne_Pustules', 'Acne_Whiteheads', 'Carcinoma', 'Eczema', 'Keratosis', 'Normal', 'Rosacea', 'Vitiligo']
Num classes: 11
Train/Val/Test sizes: 17474, 3494, 2331


,class,train,val,test
0,Acne,3759,752,501
1,Acne_Blackheads,1607,321,215
2,Acne_Cystic,250,50,34
3,Acne_Pustules,242,48,32
4,Acne_Whiteheads,72,14,10
5,Carcinoma,299,60,40
6,Eczema,970,194,129
7,Keratosis,1375,275,183
8,Normal,7610,1522,1015
9,Rosacea,774,155,103


In [6]:
# ── Imbalance handling ────────────────────────────────────────────────────
class_counts_train = np.bincount(train_targets, minlength=num_classes).astype(float)

# Smooth class weights to avoid extreme up-weighting of tiny classes
# (class weight = median_count / class_count, capped at 10x)
median_count = np.median(class_counts_train[class_counts_train > 0])
class_weights = np.where(
    class_counts_train > 0,
    np.minimum(median_count / np.maximum(class_counts_train, 1), 10.0),
    0.0,
)

print('Train class counts:', class_counts_train.astype(int).tolist())
print('Class weights     :', class_weights.round(3).tolist())

sample_weights = class_weights[train_targets]
train_sampler = None
train_shuffle = True
if cfg.use_weighted_sampler:
    train_sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )
    train_shuffle = False

_nw = cfg.num_workers
train_loader = DataLoader(train_subset, batch_size=cfg.batch_size,
                          shuffle=train_shuffle, sampler=train_sampler,
                          num_workers=_nw, pin_memory=(DEVICE=='cuda'),
                          persistent_workers=(_nw > 0))
val_loader   = DataLoader(val_subset,   batch_size=cfg.batch_size,
                          shuffle=False, num_workers=_nw,
                          pin_memory=(DEVICE=='cuda'),
                          persistent_workers=(_nw > 0))
test_loader  = DataLoader(test_subset,  batch_size=cfg.batch_size,
                          shuffle=False, num_workers=_nw,
                          pin_memory=(DEVICE=='cuda'),
                          persistent_workers=(_nw > 0))

print(f'Loaders ready | train batches={len(train_loader)} '
      f'val batches={len(val_loader)} test batches={len(test_loader)}')


Train class counts: [3759, 1607, 250, 242, 72, 299, 970, 1375, 7610, 774, 516]
Class weights     : [0.206, 0.482, 3.096, 3.198, 10.0, 2.589, 0.798, 0.563, 0.102, 1.0, 1.5]
Loaders ready | train batches=547 val batches=110 test batches=73


In [7]:
# ── Focal Loss ────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    '''Focal Loss for multi-class classification.
    Works with class weights and label smoothing.
    '''
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma           = gamma
        self.weight          = weight
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(
            logits, targets,
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction='none',
        )
        pt = torch.exp(-F.cross_entropy(logits, targets, reduction='none'))
        return ((1 - pt) ** self.gamma * ce).mean()


# ── MixUp helper ──────────────────────────────────────────────────────────
def mixup_data(x, y, alpha=0.3):
    '''Returns mixed inputs, pairs of targets, and lambda.'''
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    bs  = x.size(0)
    idx = torch.randperm(bs, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# ── Model: EfficientNet-B2 ─────────────────────────────────────────────────
weights = models.EfficientNet_B2_Weights.IMAGENET1K_V1
model   = models.efficientnet_b2(weights=weights)

# Richer head: Dropout → BN → Linear (→ classes)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features, 512),
    nn.BatchNorm1d(512),
    nn.SiLU(),
    nn.Dropout(p=0.2),
    nn.Linear(512, num_classes),
)
model = model.to(DEVICE)

# Stage 1: freeze backbone, train head only
for p in model.features.parameters():
    p.requires_grad = False

# ── Loss ──────────────────────────────────────────────────────────────────
_cw = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE) \
      if cfg.use_class_weighted_loss else None

if cfg.use_focal_loss:
    criterion = FocalLoss(gamma=cfg.focal_gamma, weight=_cw,
                          label_smoothing=cfg.label_smoothing)
    print('Using Focal Loss (gamma={})'.format(cfg.focal_gamma))
else:
    criterion = nn.CrossEntropyLoss(weight=_cw, label_smoothing=cfg.label_smoothing)
    print('Using CrossEntropyLoss')

# ── Optimiser + Scheduler ─────────────────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr,
                               weight_decay=cfg.weight_decay)

# Warm-up then cosine anneal
def lr_lambda(epoch):
    if epoch < cfg.warmup_epochs:
        return float(epoch + 1) / float(cfg.warmup_epochs)
    progress = (epoch - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Fixed deprecation warnings: use torch.amp instead of torch.cuda.amp
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

print(model.classifier)
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal params: {total_params/1e6:.1f}M | Trainable: {trainable_params/1e6:.1f}M')


Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /home/dev/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:05<00:00, 6.54MB/s]

Using Focal Loss (gamma=2.0)
Sequential(
  (0): Dropout(p=0.3, inplace=True)
  (1): Linear(in_features=1408, out_features=512, bias=True)
  (2): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): SiLU()
  (4): Dropout(p=0.2, inplace=False)
  (5): Linear(in_features=512, out_features=11, bias=True)
)

Total params: 8.4M | Trainable: 0.7M


In [8]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None,
              mixup_alpha=0.0):
    is_train = optimizer is not None
    model.train(is_train)

    running_loss, running_correct, total = 0.0, 0, 0
    y_true, y_pred = [], []

    pbar = tqdm(loader, desc='train' if is_train else 'eval ', leave=False)
    for x, y in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)
            # MixUp only during training
            x, y_a, y_b, lam = mixup_data(x, y, alpha=mixup_alpha)
        else:
            y_a, y_b, lam = y, y, 1.0

        with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
            logits = model(x)
            if is_train and mixup_alpha > 0:
                loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
            else:
                loss = criterion(logits, y_a)

        if is_train:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            scaler.step(optimizer)
            scaler.update()

        preds = logits.argmax(dim=1)
        running_loss    += loss.item() * x.size(0)
        running_correct += (preds == y_a).sum().item()
        total           += x.size(0)

        y_true.append(y_a.detach().cpu())
        y_pred.append(preds.detach().cpu())

        pbar.set_postfix(loss=f'{running_loss/total:.4f}',
                         acc=f'{running_correct/total:.4f}')

    epoch_loss = running_loss / max(total, 1)
    epoch_acc  = running_correct / max(total, 1)
    y_true = torch.cat(y_true).numpy() if y_true else np.array([])
    y_pred = torch.cat(y_pred).numpy() if y_pred else np.array([])
    return epoch_loss, epoch_acc, y_true, y_pred


def per_class_accuracy(y_true, y_pred, n):
    return {c: float((y_pred[y_true == c] == c).mean())
            if (y_true == c).sum() > 0 else float('nan')
            for c in range(n)}


def plot_confusion_matrix(y_true, y_pred, names, save_path=None):
    if not HAS_SKLEARN:
        return
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('Confusion Matrix')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


print('Helper functions defined.')


Helper functions defined.


In [ ]:
history = []
best_state = None
best_val_acc = -np.inf
patience_counter = 0
backbone_unfrozen = False

start = time.time()
for epoch in range(1, cfg.epochs + 1):

    # ── Stage 2: unfreeze backbone after warmup ──────────────────────────
    if epoch == cfg.freeze_backbone_epochs + 1 and not backbone_unfrozen:
        for p in model.features.parameters():
            p.requires_grad = True
        # Lower LR for fine-tuning backbone; head already has higher LR context
        optimizer = torch.optim.AdamW([
            {'params': model.features.parameters(),   'lr': cfg.lr * 0.1},
            {'params': model.classifier.parameters(), 'lr': cfg.lr * 0.3},
        ], weight_decay=cfg.weight_decay)
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        backbone_unfrozen = True
        total_params     = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'Backbone unfrozen | trainable: {trainable_params/1e6:.1f}M / {total_params/1e6:.1f}M')

    train_loss, train_acc, _, _ = run_epoch(
        model, train_loader, criterion, optimizer, scaler,
        mixup_alpha=cfg.mixup_alpha)
    val_loss, val_acc, yv, pv   = run_epoch(
        model, val_loader, criterion)

    scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']
    row = dict(epoch=epoch, train_loss=train_loss, train_acc=train_acc,
               val_loss=val_loss, val_acc=val_acc, lr=current_lr)
    history.append(row)

    print(
        f'Epoch {epoch:02d}/{cfg.epochs} | '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f} | lr={current_lr:.2e}'
    )

    # ── Checkpoint ──────────────────────────────────────────────────────
    if (val_acc - best_val_acc) > cfg.min_delta:
        best_val_acc = val_acc
        best_state = {
            'model':        copy.deepcopy(model.state_dict()),
            'epoch':        epoch,
            'val_acc':      val_acc,
            'config':       asdict(cfg),
            'class_names':  class_names,
            'class_to_idx': base_dataset.class_to_idx,
        }
        torch.save(best_state, RUN_DIR / 'best_model.pt')
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= cfg.patience:
        print(f'Early stopping at epoch {epoch}.')
        break

elapsed = time.time() - start
print(f'\nTraining done in {elapsed/60:.1f} min | Best val acc: {best_val_acc:.4f}')

history_df = pd.DataFrame(history)
history_df.to_csv(RUN_DIR / 'train_history.csv', index=False)
display(history_df.tail(10))


train:   0%|          | 0/547 [00:00<?, ?it/s]

eval :   0%|          | 0/110 [00:00<?, ?it/s]

Epoch 01/40 | train_loss=1.9906 train_acc=0.2264 | val_loss=0.7032 val_acc=0.1663 | lr=2.00e-04


train:   0%|          | 0/547 [00:00<?, ?it/s]

eval :   0%|          | 0/110 [00:00<?, ?it/s]

Epoch 02/40 | train_loss=1.6180 train_acc=0.2792 | val_loss=0.5488 val_acc=0.4473 | lr=3.00e-04
Backbone unfrozen | trainable: 8.4M / 8.4M


train:   0%|          | 0/547 [00:00<?, ?it/s]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_df['epoch'], history_df['train_loss'], label='Train')
axes[0].plot(history_df['epoch'], history_df['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['train_acc'], label='Train')
axes[1].plot(history_df['epoch'], history_df['val_acc'],   label='Val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.tight_layout()
plt.savefig(RUN_DIR / 'training_curves.png', dpi=150)
plt.show()


In [ ]:
# Load best checkpoint
ckpt = torch.load(RUN_DIR / 'best_model.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded best model from epoch {ckpt["epoch"]} (val_acc={ckpt["val_acc"]:.4f})')

test_loss, test_acc, yt, pt = run_epoch(model, test_loader, criterion)
print(f'Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}')

# Per-class accuracy
pc_acc = per_class_accuracy(yt, pt, num_classes)
pc_df = pd.DataFrame({
    'class':          class_names,
    'test_class_acc': [pc_acc[i] for i in range(num_classes)],
    'n_test':         np.bincount(test_targets, minlength=num_classes),
})
pc_df['test_class_acc'] = pc_df['test_class_acc'].round(4)
display(pc_df)
pc_df.to_csv(RUN_DIR / 'test_per_class_accuracy.csv', index=False)

# Full classification report
if HAS_SKLEARN:
    print('\nClassification Report:')
    print(classification_report(yt, pt, target_names=class_names, digits=3))

    report_dict = classification_report(yt, pt, target_names=class_names,
                                        digits=3, output_dict=True)
    pd.DataFrame(report_dict).T.to_csv(RUN_DIR / 'classification_report.csv')

# Confusion matrix
plot_confusion_matrix(yt, pt, class_names, save_path=RUN_DIR / 'confusion_matrix.png')


In [ ]:
# ── Production inference artifact ────────────────────────────────────────
export = {
    'model_state_dict': model.state_dict(),
    'class_names':      class_names,
    'class_to_idx':     base_dataset.class_to_idx,
    'img_size':         cfg.img_size,
    'normalize_mean':   MEAN,
    'normalize_std':    STD,
    'architecture':     'efficientnet_b2',
    'val_acc':          best_val_acc,
    'test_acc':         float(test_acc),
}
torch.save(export, RUN_DIR / 'model_inference.pt')

with open(RUN_DIR / 'config.json', 'w') as f:
    json.dump(asdict(cfg), f, indent=2)

print('Saved artifacts:')
for p in sorted(RUN_DIR.glob('*')):
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name:<40}  {size_kb:>8.1f} KB')


In [ ]:
# ── Download model from Colab ─────────────────────────────────────────────
# Run this cell only on Google Colab to download the trained model.

if IS_COLAB:
    from google.colab import files
    import shutil, zipfile

    # Bundle all artifacts into a zip for one-click download
    bundle_path = f'/content/{cfg.run_name}_artifacts.zip'
    with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(RUN_DIR.glob('*')):
            zf.write(p, arcname=p.name)
    print(f'Downloading {bundle_path} ...')
    files.download(bundle_path)
else:
    print('Not running on Colab — artifacts are already at:')
    print(f'  {RUN_DIR}')
    print()
    print('Copy them to your local machine with:')
    print(f'  scp -r <user>@<host>:{RUN_DIR} .')


In [ ]:
# ── Inference helper (with TTA) ───────────────────────────────────────────

def predict_image(image_path: str, checkpoint_path: str = None,
                  use_tta: bool = True):
    '''Predict skin condition from a single image.

    Args:
        image_path:      Path to the image file.
        checkpoint_path: Optional explicit path to model_inference.pt.
        use_tta:         Whether to average predictions over TTA views.

    Returns:
        dict with pred_class, confidence, and top3 list.
    '''
    checkpoint_path = checkpoint_path or str(RUN_DIR / 'model_inference.pt')
    blob = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

    # Rebuild model
    net = models.efficientnet_b2(weights=None)
    in_features = net.classifier[1].in_features
    net.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.SiLU(),
        nn.Dropout(p=0.2),
        nn.Linear(512, len(blob['class_names'])),
    )
    net.load_state_dict(blob['model_state_dict'])
    net = net.to(DEVICE).eval()

    im = Image.open(image_path).convert('RGB')

    views = tta_tfms if use_tta else [eval_tfms]
    all_probs = []
    with torch.no_grad(), torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
        for tfm in views:
            x = tfm(im).unsqueeze(0).to(DEVICE)
            probs = torch.softmax(net(x), dim=1)[0]
            all_probs.append(probs)

    avg_probs = torch.stack(all_probs).mean(dim=0)
    conf, idx = torch.max(avg_probs, dim=0)
    names = blob['class_names']

    top3 = sorted(enumerate(avg_probs.cpu().tolist()),
                  key=lambda z: z[1], reverse=True)[:3]

    return {
        'pred_class':  names[idx.item()],
        'confidence':  round(float(conf.item()), 4),
        'tta_views':   len(views),
        'top3': [(names[i], round(p, 4)) for i, p in top3],
    }


# Example:
# result = predict_image('path/to/skin_image.jpg', use_tta=True)
# print(result)
